In [29]:
from __future__ import annotations

from dataclasses import dataclass
import numpy as np
import pandas as pd


MINUTES_IN_YEAR = 525_600
N30 = 30 * 24 * 60  # 43,200 minutes


def _require(cond: bool, msg: str) -> None:
    if not cond:
        raise ValueError(msg)


@dataclass(frozen=True)
class PreparedChain:
    """
    Clean boundary object: a single-expiration option chain prepared for VIX math.

    - Calls/Puts are indexed by strike with columns: bid, ask, mid.
    - strikes is the sorted union of strikes across both sides.
    """
    calls: pd.DataFrame
    puts: pd.DataFrame
    strikes: np.ndarray


def validate_chain_for_vix(chain: pd.DataFrame) -> None:
    """
    Optional validation for your upstream pipeline.

    This checks only "structural" assumptions so the calculation stays deterministic:
      - required columns exist
      - cp_flag is {C,P}
      - no duplicates per (cp_flag, strike)
      - finite/nonnegative quotes and bid <= ask
      - strikes finite and > 0

    The official methodology notes that series with null quotes or bid>ask are not candidates
    (and K0 null/bid>ask makes the index not calculable), so we treat those as invalid here. :contentReference[oaicite:1]{index=1}
    """
    required = {"strike", "cp_flag", "bid", "ask"}
    missing = required - set(chain.columns)
    _require(not missing, f"Missing required columns: {missing}")

    df = chain[list(required)].copy()
    df["cp_flag"] = df["cp_flag"].astype(str).str.upper()

    _require(df["cp_flag"].isin(["C", "P"]).all(), "cp_flag must be 'C' or 'P'.")
    _require(np.isfinite(df["strike"]).all() and (df["strike"] > 0).all(), "strike must be finite and > 0.")
    _require(np.isfinite(df["bid"]).all() and (df["bid"] >= 0).all(), "bid must be finite and >= 0.")
    _require(np.isfinite(df["ask"]).all() and (df["ask"] >= 0).all(), "ask must be finite and >= 0.")
    _require((df["bid"] <= df["ask"]).all(), "Found bid > ask (invalid quotes).")

    dup = df.duplicated(subset=["cp_flag", "strike"], keep=False)
    _require(not dup.any(), "Duplicate rows for (cp_flag, strike); deduplicate upstream.")


def prepare_chain_for_vix(chain: pd.DataFrame) -> PreparedChain:
    """
    Minimal preparation consistent with the official methodology:

      - Remove series with null quotes (explicit step in strike selection). :contentReference[oaicite:2]{index=2}
      - Remove series where bid > ask (officially not candidates for ATM and can make index incalculable). :contentReference[oaicite:3]{index=3}
      - Compute mid = (bid + ask)/2
      - Split into calls/puts indexed by strike (uniqueness enforced via verify_integrity=True)

    No "extra" sanity checks beyond what’s needed to make the downstream calculation deterministic.
    """
    cols = ["strike", "cp_flag", "bid", "ask"]
    missing = set(cols) - set(chain.columns)
    _require(not missing, f"Missing required columns: {missing}")

    df = chain.loc[:, cols].copy()
    df["cp_flag"] = df["cp_flag"].astype(str).str.upper()
    df = df[df["cp_flag"].isin(["C", "P"])]

    # Official step: remove null quotes. :contentReference[oaicite:4]{index=4}
    df = df.dropna(subset=["bid", "ask", "strike"])

    # Exclude invalid markets (bid > ask) per the official candidate rules. :contentReference[oaicite:5]{index=5}
    df = df[df["bid"] <= df["ask"]]

    df["mid"] = 0.5 * (df["bid"] + df["ask"])

    calls = (
        df[df["cp_flag"] == "C"]
        .set_index("strike", verify_integrity=True)[["bid", "ask", "mid"]]
        .sort_index()
    )
    puts = (
        df[df["cp_flag"] == "P"]
        .set_index("strike", verify_integrity=True)[["bid", "ask", "mid"]]
        .sort_index()
    )

    strikes = np.sort(df["strike"].unique())
    return PreparedChain(calls=calls, puts=puts, strikes=strikes)


@dataclass(frozen=True)
class TermResult:
    minutes_to_expiry: int
    T: float
    R: float
    K_atm: float
    F: float
    K0: float
    sigma2: float
    sum_piece: float
    qk: pd.DataFrame  # columns: strike, Q


def _select_K_atm_and_forward(prep: PreparedChain, R: float, T: float) -> tuple[float, float]:
    """
    Cboe step (Forward Price and K0):
      - K_atm is the strike minimizing |C_mid - P_mid| (tie -> lowest strike)
      - F = K + exp(RT) * (C_mid - P_mid)

    Note: the math methodology excludes series with null quotes or bid>ask from ATM candidacy;
    prepare_chain_for_vix() already filtered those. :contentReference[oaicite:6]{index=6}
    """
    common = prep.calls.join(prep.puts, how="inner", lsuffix="_C", rsuffix="_P")
    _require(len(common) > 0, "No strikes with both call and put quotes; cannot infer forward.")

    tmp = pd.DataFrame(
        {
            "strike": common.index.to_numpy(dtype=float),
            "abs_diff": (common["mid_C"] - common["mid_P"]).abs().to_numpy(),
        }
    ).sort_values(["abs_diff", "strike"], ascending=[True, True])

    K_atm = float(tmp.iloc[0]["strike"])
    row = common.loc[K_atm]
    F = float(K_atm + np.exp(R * T) * (row["mid_C"] - row["mid_P"]))
    return K_atm, F


def _strike_at_or_below(strikes: np.ndarray, x: float) -> float:
    leq = strikes[strikes <= x]
    _require(len(leq) > 0, "No strike <= forward; check strike grid / forward inference.")
    return float(leq[-1])


def _select_wing_two_consecutive_zero_bid_or_ask(wing: pd.DataFrame, *, direction: str) -> pd.DataFrame:
    """
    Cboe strike selection (current language):

      - Exclude any OTM option with bid == 0 OR ask == 0
      - Once two consecutive strikes are found with zero bid OR zero ask,
        exclude those observed options and consider no strikes beyond.

    This reflects the post–Feb 10, 2025 update in the official math methodology. :contentReference[oaicite:7]{index=7}
    """
    if wing.empty:
        return wing

    ordered = wing.sort_index(ascending=(direction == "up"))
    is_zero = (ordered["bid"].to_numpy() == 0) | (ordered["ask"].to_numpy() == 0)

    cut = None
    for i in range(len(is_zero) - 1):
        if is_zero[i] and is_zero[i + 1]:
            cut = i
            break

    if cut is None:
        # Keep all non-zero quotes
        return ordered.loc[~pd.Series(is_zero, index=ordered.index)]
    else:
        # Stop strictly before the first zero in the consecutive pair; also exclude any zero rows before the cut.
        kept = ordered.iloc[:cut]
        kept_zero = is_zero[:cut]
        return kept.loc[~pd.Series(kept_zero, index=kept.index)]


def _build_qk_table(prep: PreparedChain, K0: float) -> pd.DataFrame:
    """
    Cboe strike selection + pricing Q(K):

      - K < K0: OTM puts selected by the zero-quote rule
      - K > K0: OTM calls selected by the zero-quote rule
      - K = K0: include BOTH put and call; Q(K0) is their average midquote

    If all OTM puts or all OTM calls are excluded, the index cannot be calculated. :contentReference[oaicite:8]{index=8}
    """
    _require(K0 in prep.calls.index and K0 in prep.puts.index, "K0 must have both a call and a put quote.")

    puts_otm = prep.puts.loc[prep.puts.index < K0]
    calls_otm = prep.calls.loc[prep.calls.index > K0]

    puts_sel = _select_wing_two_consecutive_zero_bid_or_ask(puts_otm, direction="down")
    calls_sel = _select_wing_two_consecutive_zero_bid_or_ask(calls_otm, direction="up")

    _require(len(puts_sel) > 0, "All OTM puts excluded; volatility index cannot be calculated.")
    _require(len(calls_sel) > 0, "All OTM calls excluded; volatility index cannot be calculated.")

    q0 = float(0.5 * (prep.calls.loc[K0, "mid"] + prep.puts.loc[K0, "mid"]))

    qk = pd.concat(
        [
            pd.DataFrame({"strike": puts_sel.index.to_numpy(dtype=float), "Q": puts_sel["mid"].to_numpy(dtype=float)}),
            pd.DataFrame({"strike": np.array([K0], dtype=float), "Q": np.array([q0], dtype=float)}),
            pd.DataFrame({"strike": calls_sel.index.to_numpy(dtype=float), "Q": calls_sel["mid"].to_numpy(dtype=float)}),
        ],
        ignore_index=True,
    )

    qk = qk.groupby("strike", as_index=False)["Q"].mean().sort_values("strike").reset_index(drop=True)
    return qk


def _deltaK(strikes: np.ndarray) -> np.ndarray:
    """
    Cboe ΔK definition:
      - endpoints: adjacent difference
      - interior: (K_{i+1} - K_{i-1}) / 2
    """
    strikes = np.asarray(strikes, dtype=float)
    _require(len(strikes) >= 2, "Need at least 2 included strikes to compute ΔK.")

    dK = np.empty_like(strikes)
    dK[0] = strikes[1] - strikes[0]
    dK[-1] = strikes[-1] - strikes[-2]
    dK[1:-1] = 0.5 * (strikes[2:] - strikes[:-2])
    return dK


def compute_single_term_variance(prep: PreparedChain, *, R: float, minutes_to_expiry: int) -> TermResult:
    """
    Cboe single-term variance for one expiration.
    """
    _require(minutes_to_expiry > 0, "minutes_to_expiry must be positive.")
    T = minutes_to_expiry / MINUTES_IN_YEAR

    K_atm, F = _select_K_atm_and_forward(prep, R=R, T=T)
    K0 = _strike_at_or_below(prep.strikes, F)

    qk = _build_qk_table(prep, K0=K0)

    strikes = qk["strike"].to_numpy(dtype=float)
    Q = qk["Q"].to_numpy(dtype=float)
    dK = _deltaK(strikes)

    disc = np.exp(R * T)
    sum_piece = float(np.sum((dK / (strikes ** 2)) * disc * Q))

    # σ^2 = (2/T)*Σ[...] - (1/T)*(F/K0 - 1)^2
    sigma2 = float((2.0 / T) * sum_piece - (1.0 / T) * ((F / K0) - 1.0) ** 2)

    return TermResult(
        minutes_to_expiry=minutes_to_expiry,
        T=T,
        R=R,
        K_atm=K_atm,
        F=F,
        K0=K0,
        sigma2=sigma2,
        sum_piece=sum_piece,
        qk=qk,
    )

def interpolate_vix_30day(near: TermResult, next_: TermResult, target_minutes: int = N30) -> float:
    """
    Cboe constant-maturity interpolation to 30 days (in minutes), then VIX = 100*sqrt(var_30).
    """
    N1, N2 = near.minutes_to_expiry, next_.minutes_to_expiry
    _require(N1 < target_minutes < N2, "Need N1 < target_minutes < N2 for interpolation.")

    w1 = (N2 - target_minutes) / (N2 - N1)
    w2 = (target_minutes - N1) / (N2 - N1)

    var_30 = (w1 * near.T * near.sigma2 + w2 * next_.T * next_.sigma2) * (MINUTES_IN_YEAR / target_minutes)
    return float(100.0 * np.sqrt(var_30))

In [45]:
import numpy as np
import pandas as pd
import wrds
from zoneinfo import ZoneInfo

SPX_SECID = 108105
NY = ZoneInfo("America/New_York")

def _table_for_date(dt: pd.Timestamp) -> str:
    return f"optionm_all.opprcd{dt.year}"

def fetch_opprcd_slice(db: wrds.Connection, date: str,
                       exdate_min_days: int = 20,
                       exdate_max_days: int = 45) -> pd.DataFrame:
    d = pd.Timestamp(date)
    table = _table_for_date(d)
    ex_min = (d + pd.Timedelta(days=exdate_min_days)).date()
    ex_max = (d + pd.Timedelta(days=exdate_max_days)).date()

    sql = f"""
        SELECT
            secid, date, exdate, cp_flag, strike_price, best_bid, best_offer,
            am_settlement, ss_flag, contract_size, expiry_indicator
        FROM {table}
        WHERE secid = {SPX_SECID}
          AND date = '{d.date()}'
          AND exdate BETWEEN '{ex_min}' AND '{ex_max}'
          AND contract_size = 100
          AND ss_flag = '0'
          AND cp_flag IN ('C','P')
          AND best_bid IS NOT NULL
          AND best_offer IS NOT NULL
          AND strike_price IS NOT NULL
          AND best_bid <= best_offer
    """
    return db.raw_sql(sql)

def fetch_zero_curve(db: wrds.Connection, date: str) -> pd.DataFrame:
    d = pd.Timestamp(date).date()
    sql = f"""
        SELECT date, days, rate
        FROM optionm_all.zerocd
        WHERE date = '{d}'
        ORDER BY days
    """
    return db.raw_sql(sql)

def minutes_to_expiry(asof_date: str, exdate: pd.Timestamp, am_settlement: float) -> int:
    asof = pd.Timestamp(asof_date).tz_localize(NY) + pd.Timedelta(hours=16)
    exp = (
        pd.Timestamp(exdate).tz_localize(NY)
        + (pd.Timedelta(hours=9, minutes=30) if int(am_settlement) == 1 else pd.Timedelta(hours=16))
    )
    return int((exp - asof).total_seconds() // 60)

def pick_near_next_terms(df_day: pd.DataFrame, asof_date: str, target_minutes: int):
    df = df_day.copy()
    df["exdate"] = pd.to_datetime(df["exdate"])

    # VIX universe: AM-settled SPX + PM-settled SPXW end-of-week,
    # excluding PM expiries that coincide with AM expiries.
    am_expiries = set(df.loc[df["am_settlement"].fillna(0).astype(int) == 1, "exdate"].dt.date.unique())

    am = df["am_settlement"].fillna(0).astype(int)
    is_am = am == 1
    is_pm_eow = (am == 0) & (df["exdate"].dt.weekday == 4)  # PoC: Friday-only EOW
    is_same_date_pm = (am == 0) & (df["exdate"].dt.date.isin(am_expiries))

    df = df[(is_am | is_pm_eow) & (~is_same_date_pm)]

    series = df[["exdate", "am_settlement"]].drop_duplicates().copy()
    series["N"] = [
        minutes_to_expiry(asof_date, ex, am_)
        for ex, am_ in zip(series["exdate"], series["am_settlement"])
    ]
    series = series[series["N"] > 0].sort_values("N")

    near = series[series["N"] < target_minutes].tail(1)
    nxt  = series[series["N"] > target_minutes].head(1)
    if near.empty or nxt.empty:
        raise ValueError("Could not find Near/Next terms bracketing 30 days in the pulled expiry window.")

    e1 = (pd.Timestamp(near.iloc[0]["exdate"]), int(near.iloc[0]["am_settlement"]))
    e2 = (pd.Timestamp(nxt.iloc[0]["exdate"]), int(nxt.iloc[0]["am_settlement"]))
    return e1, e2

def interp_rate_from_zerocd(zc: pd.DataFrame, days_to_exp: float) -> float:
    """
    Linearly interpolate the OptionMetrics zero curve by days.
    Note: `rate` is continuously-compounded, but appears to be in *percent* units on WRDS for your sample.
    """
    x = zc["days"].to_numpy(dtype=float)
    y = zc["rate"].to_numpy(dtype=float)
    return float(np.interp(days_to_exp, x, y))

def rate_to_decimal(r: float) -> float:
    return r / 100.0 if r > 1.0 else r

def build_chain(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "strike": df["strike_price"].astype(float) / 1000.0,
        "cp_flag": df["cp_flag"].astype(str),
        "bid": df["best_bid"].astype(float),
        "ask": df["best_offer"].astype(float),
    })

# ----------------- run the PoC -----------------
asof_date = "2025-03-10"
target_minutes = 30 * 24 * 60

db = wrds.Connection()

df = fetch_opprcd_slice(db, asof_date, exdate_min_days=20, exdate_max_days=45)
zc = fetch_zero_curve(db, asof_date)

(ex1, am1), (ex2, am2) = pick_near_next_terms(df, asof_date, target_minutes)

df1 = df[(pd.to_datetime(df["exdate"]) == ex1) & (df["am_settlement"].fillna(0).astype(int) == am1)]
df2 = df[(pd.to_datetime(df["exdate"]) == ex2) & (df["am_settlement"].fillna(0).astype(int) == am2)]

chain1 = build_chain(df1)
chain2 = build_chain(df2)

N1 = minutes_to_expiry(asof_date, ex1, am1)
N2 = minutes_to_expiry(asof_date, ex2, am2)

d = pd.Timestamp(asof_date)
days1 = (ex1.normalize() - d.normalize()).days
days2 = (ex2.normalize() - d.normalize()).days

R1 = rate_to_decimal(interp_rate_from_zerocd(zc, days1))
R2 = rate_to_decimal(interp_rate_from_zerocd(zc, days2))

prep1 = prepare_chain_for_vix(chain1)
prep2 = prepare_chain_for_vix(chain2)

near = compute_single_term_variance(prep1, R=R1, minutes_to_expiry=N1)
nxt  = compute_single_term_variance(prep2, R=R2, minutes_to_expiry=N2)

vix = interpolate_vix_30day(near, nxt, target_minutes=target_minutes)
print("VIX-like (EOD PoC):", vix)

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: fe_sendauth: no password supplied

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [31]:
import pandas as pd
import numpy as np

def inspect_raw_slice(df: pd.DataFrame):
    print("Rows:", len(df))
    print("Unique secid:", df["secid"].nunique(), "->", sorted(df["secid"].dropna().unique())[:5])
    print("Date unique:", df["date"].nunique(), "->", df["date"].unique()[:5])
    print("Exdates:", df["exdate"].nunique())
    print("\ncp_flag counts:\n", df["cp_flag"].value_counts(dropna=False))
    print("\nam_settlement counts:\n", df["am_settlement"].value_counts(dropna=False))
    print("\nexpiry_indicator counts:\n", df["expiry_indicator"].value_counts(dropna=False))
    print("\nroot counts (top 10):\n", df["root"].value_counts(dropna=False).head(10))
    print("\ncontract_size counts:\n", df["contract_size"].value_counts(dropna=False).head(10))
    print("\nss_flag counts:\n", df["ss_flag"].value_counts(dropna=False).head(10))

    # Strike scaling check
    s = df["strike_price"].astype(float)
    print("\nstrike_price summary (raw, scaled x1000):")
    print(s.describe(percentiles=[.01,.05,.5,.95,.99]))
    print("Example strikes (converted):", (s.dropna().sort_values().head(3) / 1000).tolist(), "...",
          (s.dropna().sort_values().tail(3) / 1000).tolist())

    # Bid/ask health
    b = df["best_bid"].astype(float)
    a = df["best_offer"].astype(float)
    print("\nbest_bid summary:")
    print(b.describe(percentiles=[.01,.05,.5,.95,.99]))
    print("\nbest_offer summary:")
    print(a.describe(percentiles=[.01,.05,.5,.95,.99]))
    print("\nBad markets (bid>ask):", int((b > a).sum()))
    print("Zero bid rows:", int((b == 0).sum()), "Zero ask rows:", int((a == 0).sum()))
    
inspect_raw_slice(df)

Rows: 5462
Unique secid: 1 -> [np.float64(108105.0)]
Date unique: 1 -> <StringArray>
['2025-03-10']
Length: 1, dtype: string
Exdates: 14

cp_flag counts:
 cp_flag
C    2731
P    2731
Name: count, dtype: Int64

am_settlement counts:
 am_settlement
0.0    4616
1.0     846
Name: count, dtype: Int64

expiry_indicator counts:
 expiry_indicator
<NA>    2740
w       1782
m        940
Name: count, dtype: Int64

root counts (top 10):
 root
<NA>    5462
Name: count, dtype: Int64

contract_size counts:
 contract_size
100.0    5462
Name: count, dtype: Int64

ss_flag counts:
 ss_flag
0    5462
Name: count, dtype: Int64

strike_price summary (raw, scaled x1000):
count    5.462000e+03
mean     5.500205e+06
std      1.115592e+06
min      2.000000e+05
1%       1.800000e+06
5%       3.202500e+06
50%      5.690000e+06
95%      6.800000e+06
99%      8.400000e+06
max      1.100000e+07
Name: strike_price, dtype: float64
Example strikes (converted): [200.0, 200.0, 200.0] ... [11000.0, 11000.0, 11000.0]

best

In [32]:
import pandas as pd

def inspect_expiries(df):
    x = df.copy()
    x["exdate"] = pd.to_datetime(x["exdate"])
    x["am"] = x["am_settlement"].fillna(0).astype(int)

    exp = (x[["exdate","am","expiry_indicator"]]
           .drop_duplicates()
           .assign(weekday=lambda t: t["exdate"].dt.day_name())
           .sort_values(["exdate","am"]))

    print("Unique expiry series (exdate, AM/PM):", len(exp))
    print("\nCounts by AM/PM and weekday:")
    print(exp.groupby(["am","weekday"]).size().unstack(fill_value=0))

    print("\nSample expiry series:")
    print(exp.head(20).to_string(index=False))

inspect_expiries(df)

Unique expiry series (exdate, AM/PM): 15

Counts by AM/PM and weekday:
weekday  Friday  Monday  Thursday  Tuesday  Wednesday
am                                                   
0             2       3         3        3          3
1             0       0         1        0          0

Sample expiry series:
    exdate  am expiry_indicator   weekday
2025-03-31   0                m    Monday
2025-04-01   0             <NA>   Tuesday
2025-04-02   0                w Wednesday
2025-04-03   0             <NA>  Thursday
2025-04-04   0                w    Friday
2025-04-07   0                w    Monday
2025-04-08   0             <NA>   Tuesday
2025-04-09   0                w Wednesday
2025-04-10   0             <NA>  Thursday
2025-04-11   0                w    Friday
2025-04-14   0                w    Monday
2025-04-15   0             <NA>   Tuesday
2025-04-16   0                w Wednesday
2025-04-17   0             <NA>  Thursday
2025-04-17   1             <NA>  Thursday


In [33]:
def vix_universe_filter(df):
    x = df.copy()
    x["exdate"] = pd.to_datetime(x["exdate"])
    am = x["am_settlement"].fillna(0).astype(int)

    am_expiries = set(x.loc[am == 1, "exdate"].dt.date.unique())
    is_am = (am == 1)
    is_pm_eow = (am == 0) & (x["exdate"].dt.weekday == 4)  # Fri
    is_same_date_pm = (am == 0) & (x["exdate"].dt.date.isin(am_expiries))

    y = x[(is_am | is_pm_eow) & (~is_same_date_pm)].copy()
    return y

df_u = vix_universe_filter(df)
print("Rows before:", len(df), "Rows after universe filter:", len(df_u))
inspect_expiries(df_u)

Rows before: 5462 Rows after universe filter: 1702
Unique expiry series (exdate, AM/PM): 3

Counts by AM/PM and weekday:
weekday  Friday  Thursday
am                       
0             2         0
1             0         1

Sample expiry series:
    exdate  am expiry_indicator  weekday
2025-04-04   0                w   Friday
2025-04-11   0                w   Friday
2025-04-17   1             <NA> Thursday


In [34]:
from zoneinfo import ZoneInfo
NY = ZoneInfo("America/New_York")
TARGET = 30*24*60

def minutes_to_expiry(asof_date, exdate, am_settlement):
    asof = pd.Timestamp(asof_date).tz_localize(NY) + pd.Timedelta(hours=16)
    exp = pd.Timestamp(exdate).tz_localize(NY) + (pd.Timedelta(hours=9, minutes=30) if int(am_settlement)==1 else pd.Timedelta(hours=16))
    return int((exp - asof).total_seconds() // 60)

def pick_terms(df_u, asof_date):
    x = df_u.copy()
    x["exdate"] = pd.to_datetime(x["exdate"])
    x["am"] = x["am_settlement"].fillna(0).astype(int)

    series = (x[["exdate","am"]].drop_duplicates()
              .assign(N=lambda t: t.apply(lambda r: minutes_to_expiry(asof_date, r["exdate"], r["am"]), axis=1))
              .sort_values("N"))
    series = series[series["N"] > 0]

    print(series.head(15).to_string(index=False))
    near = series[series["N"] < TARGET].tail(1)
    nxt  = series[series["N"] > TARGET].head(1)

    print("\nChosen near:\n", near.to_string(index=False))
    print("Chosen next:\n", nxt.to_string(index=False))

pick_terms(df_u, "2025-03-10")

    exdate  am     N
2025-04-04   0 36000
2025-04-11   0 46080
2025-04-17   1 54330

Chosen near:
     exdate  am     N
2025-04-04   0 36000
Chosen next:
     exdate  am     N
2025-04-11   0 46080


In [35]:
def term_df(df_u, exdate, am):
    x = df_u.copy()
    x["exdate"] = pd.to_datetime(x["exdate"])
    return x[(x["exdate"] == pd.Timestamp(exdate)) & (x["am_settlement"].fillna(0).astype(int) == int(am))].copy()

def inspect_zero_pattern(df_term):
    t = pd.DataFrame({
        "strike": df_term["strike_price"].astype(float)/1000.0,
        "cp": df_term["cp_flag"].astype(str),
        "bid": df_term["best_bid"].astype(float),
        "ask": df_term["best_offer"].astype(float),
    })
    # Look at OTM-wing zero-bid density by side; we don't know K0 yet, but this still shows whether zeros exist in the tails.
    for side in ["P","C"]:
        s = t[t.cp == side].sort_values("strike")
        z = (s.bid == 0) | (s.ask == 0)
        # show the last 30 strikes for puts and first 30 for calls to see zeros in tails
        tail = s.tail(40).assign(z=z.tail(40).values)
        head = s.head(40).assign(z=z.head(40).values)
        print(f"\n{side} head (lowest strikes):")
        print(head[["strike","bid","ask","z"]].head(15).to_string(index=False))
        print(f"\n{side} tail (highest strikes):")
        print(tail[["strike","bid","ask","z"]].tail(15).to_string(index=False))

# Example: after you determine chosen near/next:
# df_near = term_df(df_u, ex1, am1)
# inspect_zero_pattern(df_near)

In [36]:
import pandas as pd
import numpy as np

asof_date = "2025-03-10"

# df_u is your universe-filtered DataFrame already
df_u = df_u.copy()
df_u["exdate"] = pd.to_datetime(df_u["exdate"])
df_u["am"] = df_u["am_settlement"].fillna(0).astype(int)

near_exdate = pd.Timestamp("2025-04-04")
near_am = 0
next_exdate = pd.Timestamp("2025-04-11")
next_am = 0

def build_chain(df_term: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "strike": df_term["strike_price"].astype(float) / 1000.0,
        "cp_flag": df_term["cp_flag"].astype(str),
        "bid": df_term["best_bid"].astype(float),
        "ask": df_term["best_offer"].astype(float),
    })

def inspect_term(df_term: pd.DataFrame, label: str):
    chain = build_chain(df_term)
    calls = chain[chain.cp_flag == "C"].groupby("strike").size()
    puts  = chain[chain.cp_flag == "P"].groupby("strike").size()
    common = calls.index.intersection(puts.index)

    print(f"\n== {label} ==")
    print("rows:", len(df_term))
    print("unique strikes:", chain["strike"].nunique())
    print("strikes with BOTH call & put:", len(common))
    print("min/median/max strike:", chain["strike"].min(), chain["strike"].median(), chain["strike"].max())

    zero = (chain["bid"] == 0) | (chain["ask"] == 0)
    print("zero-quote rows (bid==0 or ask==0):", int(zero.sum()))

    # show tails to see if you ever get consecutive zeros
    for side in ["P", "C"]:
        s = chain[chain.cp_flag == side].sort_values("strike")
        z = ((s["bid"] == 0) | (s["ask"] == 0)).to_numpy()
        print(f"\n{label} {side} highest-strike tail (last 20):")
        tail = s.tail(20).assign(z=z[-20:])
        print(tail[["strike","bid","ask","z"]].to_string(index=False))

# Slice the two terms
df_near = df_u[(df_u["exdate"] == near_exdate) & (df_u["am"] == near_am)]
df_next = df_u[(df_u["exdate"] == next_exdate) & (df_u["am"] == next_am)]

inspect_term(df_near, "NEAR 2025-04-04 PM")
inspect_term(df_next, "NEXT 2025-04-11 PM")

# Optional: inspect OptionMetrics forward_price column presence
# for label, dft in [("NEAR", df_near), ("NEXT", df_next)]:
#     fp = dft["forward_price"].dropna().astype(float)
#     print(f"\n{label} forward_price column describe:")
#     print(fp.describe() if len(fp) else "(empty)")


== NEAR 2025-04-04 PM ==
rows: 494
unique strikes: 247
strikes with BOTH call & put: 247
min/median/max strike: 1200.0 5770.0 7800.0
zero-quote rows (bid==0 or ask==0): 5

NEAR 2025-04-04 PM P highest-strike tail (last 20):
 strike    bid    ask     z
 6375.0  737.1  754.1 False
 6380.0  742.1  759.1 False
 6390.0  752.0  769.1 False
 6400.0  761.9  779.0 False
 6410.0  771.9  787.5 False
 6425.0  786.9  803.9 False
 6450.0  811.7  828.8 False
 6475.0  836.7  853.7 False
 6500.0  861.5  878.6 False
 6525.0  886.5  903.5 False
 6550.0  911.4  928.4 False
 6600.0  961.2  978.2 False
 6650.0 1011.1 1028.0 False
 6700.0 1060.8 1077.8 False
 6800.0 1160.6 1177.5 False
 7000.0 1359.9 1376.8 False
 7200.0 1559.2 1576.0 False
 7400.0 1758.5 1775.3 False
 7600.0 1957.8 1974.7 False
 7800.0 2157.1 2174.0 False

NEAR 2025-04-04 PM C highest-strike tail (last 20):
 strike  bid  ask     z
 6375.0 0.10 0.40 False
 6380.0 0.10 0.40 False
 6390.0 0.10 0.40 False
 6400.0 0.10 0.40 False
 6410.0 0.10 0

In [37]:
import pandas as pd
import numpy as np

def build_chain(df_term: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "strike": df_term["strike_price"].astype(float) / 1000.0,
        "cp_flag": df_term["cp_flag"].astype(str),
        "bid": df_term["best_bid"].astype(float),
        "ask": df_term["best_offer"].astype(float),
    })

def inspect_term_otm_wings(df_term: pd.DataFrame, label: str, *, asof_date: str):
    chain = build_chain(df_term)

    prep = prepare_chain_for_vix(chain)
    # Use a placeholder R for inspection (doesn't affect K0 selection much);
    # you can plug in R from zerocd later.
    tmp = compute_single_term_variance(prep, R=0.0, minutes_to_expiry=1)  # minutes doesn't matter for K0 logic much
    K0 = tmp.K0
    F  = tmp.F
    Katm = tmp.K_atm

    print(f"\n== {label} ==")
    print(f"K_atm={Katm:.1f}  F={F:.2f}  K0={K0:.1f}")

    # OTM wings relative to K0
    puts = chain[chain.cp_flag == "P"].sort_values("strike")
    calls = chain[chain.cp_flag == "C"].sort_values("strike")

    puts_otm = puts[puts["strike"] < K0]
    calls_otm = calls[calls["strike"] > K0]

    def show_tail(df_side: pd.DataFrame, side: str, which: str):
        z = (df_side["bid"] == 0) | (df_side["ask"] == 0)
        out = df_side.assign(z=z).copy()
        print(f"\n{side} OTM {which} (showing 25 rows):")
        print(out[["strike","bid","ask","z"]].to_string(index=False))

    # For OTM puts, the far wing is LOW strikes -> show HEAD
    show_tail(puts_otm.head(25), "PUT", "LOW-strike tail")
    # For OTM calls, the far wing is HIGH strikes -> show TAIL
    show_tail(calls_otm.tail(25), "CALL", "HIGH-strike tail")

    print("\nCounts:")
    print("OTM puts:", len(puts_otm), "OTM calls:", len(calls_otm),
          "zero-quote rows total:", int(((chain.bid==0)|(chain.ask==0)).sum()))

# Use your already-sliced df_near / df_next
inspect_term_otm_wings(df_near, "NEAR 2025-04-04 PM", asof_date="2025-03-10")
inspect_term_otm_wings(df_next, "NEXT 2025-04-11 PM", asof_date="2025-03-10")


== NEAR 2025-04-04 PM ==
K_atm=5625.0  F=5626.95  K0=5625.0

PUT OTM LOW-strike tail (showing 25 rows):
 strike   bid   ask     z
 1200.0  0.00  0.15  True
 1400.0  0.00  0.20  True
 1600.0  0.00  0.20  True
 1800.0  0.05  0.15 False
 2000.0  0.10  0.30 False
 2200.0  0.10  0.35 False
 2400.0  0.20  0.40 False
 2600.0  0.30  0.55 False
 2800.0  0.45  0.70 False
 3000.0  0.65  0.90 False
 3200.0  0.95  1.15 False
 3400.0  1.35  1.60 False
 3600.0  1.90  2.15 False
 3800.0  2.65  2.90 False
 4000.0  3.70  3.90 False
 4200.0  5.00  5.30 False
 4300.0  5.80  6.10 False
 4400.0  6.70  7.10 False
 4500.0  7.90  8.30 False
 4600.0  9.40  9.80 False
 4650.0 10.30 10.70 False
 4700.0 11.30 11.70 False
 4750.0 12.50 12.90 False
 4800.0 13.90 14.40 False
 4850.0 15.60 16.10 False

CALL OTM HIGH-strike tail (showing 25 rows):
 strike  bid  ask     z
 6330.0 0.20 0.45 False
 6340.0 0.15 0.45 False
 6350.0 0.15 0.40 False
 6360.0 0.15 0.40 False
 6370.0 0.10 0.40 False
 6375.0 0.10 0.40 False
 6380

In [38]:
def inspect_selected_strikes(term: TermResult, label: str):
    qk = term.qk.copy()
    strikes = qk["strike"].to_numpy()
    print(f"\n== {label} selected strikes ==")
    print("K0:", term.K0, "F:", term.F, "K_atm:", term.K_atm)
    print("Included strikes:", len(strikes))
    print("Min/Max included strike:", strikes.min(), strikes.max())
    print("First 10:", strikes[:10])
    print("Last 10:", strikes[-10:])

# After you compute near and nxt "for real" (with R and minutes_to_expiry)
inspect_selected_strikes(near, "NEAR")
inspect_selected_strikes(nxt, "NEXT")


== NEAR selected strikes ==
K0: 5625.0 F: 5626.955957120738 K_atm: 5625.0
Included strikes: 242
Min/Max included strike: 1800.0 7600.0
First 10: [1800. 2000. 2200. 2400. 2600. 2800. 3000. 3200. 3400. 3600.]
Last 10: [6500. 6525. 6550. 6600. 6650. 6700. 7000. 7200. 7400. 7600.]

== NEXT selected strikes ==
K0: 5625.0 F: 5629.447846166933 K_atm: 5630.0
Included strikes: 176
Min/Max included strike: 1400.0 6800.0
First 10: [1400. 1600. 1800. 2000. 2200. 2400. 2600. 2800. 3000. 3200.]
Last 10: [6400. 6425. 6450. 6475. 6500. 6550. 6600. 6650. 6700. 6800.]


In [39]:
import pandas as pd
import numpy as np

def where_consecutive_zero(wing: pd.DataFrame, direction: str):
    ordered = wing.sort_index(ascending=(direction == "up"))
    is_zero = (ordered["bid"].to_numpy() == 0) | (ordered["ask"].to_numpy() == 0)
    for i in range(len(is_zero)-1):
        if is_zero[i] and is_zero[i+1]:
            k_first = ordered.index[i]
            k_second = ordered.index[i+1]
            return (float(k_first), float(k_second))
    return None

def summarize_wing_stops(chain_df: pd.DataFrame):
    # expects columns: strike, cp_flag, bid, ask (your build_chain output)
    calls = chain_df[chain_df.cp_flag == "C"].set_index("strike").sort_index()
    puts  = chain_df[chain_df.cp_flag == "P"].set_index("strike").sort_index()
    prep = prepare_chain_for_vix(chain_df)
    tr = compute_single_term_variance(prep, R=0.0, minutes_to_expiry=1)
    K0 = tr.K0

    puts_otm = puts.loc[puts.index < K0]
    calls_otm = calls.loc[calls.index > K0]

    put_stop = where_consecutive_zero(puts_otm[["bid","ask"]], direction="down")
    call_stop = where_consecutive_zero(calls_otm[["bid","ask"]], direction="up")

    print("K0 =", K0)
    print("Put wing consecutive-zero stop:", put_stop)
    print("Call wing consecutive-zero stop:", call_stop)

# Example:
summarize_wing_stops(build_chain(df_near))
summarize_wing_stops(build_chain(df_next))

K0 = 5625.0
Put wing consecutive-zero stop: (1600.0, 1400.0)
Call wing consecutive-zero stop: None
K0 = 5625.0
Put wing consecutive-zero stop: None
Call wing consecutive-zero stop: (7000.0, 7200.0)


In [40]:
import numpy as np
import pandas as pd

def contribution_table(term):
    qk = term.qk.copy()
    K = qk["strike"].to_numpy(float)
    Q = qk["Q"].to_numpy(float)

    # ΔK as in your implementation
    dK = np.empty_like(K)
    dK[0] = K[1] - K[0]
    dK[-1] = K[-1] - K[-2]
    dK[1:-1] = 0.5 * (K[2:] - K[:-2])

    disc = np.exp(term.R * term.T)
    contrib = (dK / (K**2)) * disc * Q  # this is what's summed in your code

    out = pd.DataFrame({"strike": K, "Q": Q, "dK": dK, "contrib": contrib})
    out["contrib_share"] = out["contrib"] / out["contrib"].sum()
    return out.sort_values("contrib", ascending=False)

# Example usage:
ct_near = contribution_table(near)
ct_next = contribution_table(nxt)

print("Top 15 NEAR contributions:")
print(ct_near.head(15).to_string(index=False))

print("\nTop 15 NEXT contributions:")
print(ct_next.head(15).to_string(index=False))

# How much do the very low strikes matter?
print("\nNEXT: share from strikes <= 2000:",
      ct_next.loc[ct_next.strike <= 2000, "contrib_share"].sum())

Top 15 NEAR contributions:
 strike       Q    dK  contrib  contrib_share
 4000.0   3.800 200.0 0.000048       0.016948
 4200.0   5.150 150.0 0.000044       0.015625
 5590.0 128.000  10.0 0.000041       0.014615
 4500.0   8.100 100.0 0.000040       0.014272
 3800.0   2.775 200.0 0.000039       0.013713
 5560.0 117.350  10.0 0.000038       0.013544
 5550.0 114.050  10.0 0.000037       0.013211
 5540.0 110.400  10.0 0.000036       0.012834
 4400.0   6.900 100.0 0.000036       0.012716
 4600.0   9.600  75.0 0.000034       0.012140
 4850.0  15.850  50.0 0.000034       0.012021
 5510.0 101.250  10.0 0.000033       0.011899
 5500.0  98.050  10.0 0.000033       0.011565
 4300.0   5.950 100.0 0.000032       0.011482
 5600.0 131.950   7.5 0.000032       0.011259

Top 15 NEXT contributions:
 strike      Q    dK  contrib  contrib_share
 4000.0   5.35 200.0 0.000067       0.019716
 4200.0   7.05 150.0 0.000060       0.017674
 5000.0  28.55  50.0 0.000057       0.016834
 3800.0   4.05 200.0 0.000056

In [41]:
def tail_share(term, *, lo=None, hi=None):
    ct = contribution_table(term)
    if lo is not None:
        return ct.loc[ct.strike <= lo, "contrib_share"].sum()
    if hi is not None:
        return ct.loc[ct.strike >= hi, "contrib_share"].sum()
    raise ValueError("Provide lo or hi")

print("NEAR share from strikes >= 7000:", tail_share(near, hi=7000))
print("NEAR share from strikes <= 2000:", tail_share(near, lo=2000))
print("NEXT share from strikes >= 7000:", tail_share(nxt, hi=7000))

NEAR share from strikes >= 7000: 0.0007624447472037901
NEAR share from strikes <= 2000: 0.005770400008313241
NEXT share from strikes >= 7000: 0.0


In [42]:
def cumulative_by_strike(term):
    qk = term.qk.copy().sort_values("strike")
    K = qk["strike"].to_numpy(float)
    Q = qk["Q"].to_numpy(float)

    dK = np.empty_like(K)
    dK[0] = K[1] - K[0]
    dK[-1] = K[-1] - K[-2]
    dK[1:-1] = 0.5 * (K[2:] - K[:-2])

    disc = np.exp(term.R * term.T)
    contrib = (dK / (K**2)) * disc * Q
    share = contrib / contrib.sum()
    cum = np.cumsum(share)

    out = pd.DataFrame({"strike": K, "share": share, "cum_share": cum})
    return out

cum_next = cumulative_by_strike(nxt)
print(cum_next.head(12).to_string(index=False))
print("... 50% contribution reached by strike:", float(cum_next.loc[cum_next.cum_share >= 0.5, "strike"].iloc[0]))
print("... 90% contribution reached by strike:", float(cum_next.loc[cum_next.cum_share >= 0.9, "strike"].iloc[0]))

 strike    share  cum_share
 1400.0 0.003760   0.003760
 1600.0 0.003455   0.007215
 1800.0 0.003185   0.010400
 2000.0 0.003685   0.014085
 2200.0 0.003959   0.018045
 2400.0 0.004607   0.022651
 2600.0 0.005451   0.028103
 2800.0 0.006205   0.034307
 3000.0 0.007370   0.041678
 3200.0 0.008925   0.050603
 3400.0 0.010839   0.061442
 3600.0 0.013421   0.074863
... 50% contribution reached by strike: 5410.0
... 90% contribution reached by strike: 5775.0


In [43]:
print("NEXT min strike:", (df_next["strike_price"].astype(float).min() / 1000.0))
print("NEXT count bid==0 in puts:", int(((df_next["cp_flag"]=="P") & (df_next["best_bid"].astype(float)==0)).sum()))

NEXT min strike: 1400.0
NEXT count bid==0 in puts: 0


In [44]:
print("N1,N2:", near.minutes_to_expiry, nxt.minutes_to_expiry)
print("R1,R2:", near.R, nxt.R)
print("sigma2 near,next:", near.sigma2, nxt.sigma2)
print("VIX-like:", interpolate_vix_30day(near, nxt))

N1,N2: 36000 46080
R1,R2: 0.0445340425 0.04458032800000001
sigma2 near,next: 0.08208779447542963 0.07767460589749997
VIX-like: 28.058040751756224
